# **Classification: Train and Compare Different Models**


In [23]:
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter

In [24]:
X = np.load('/content/embeddings.npy')
y = np.load('/content/labels.npy')

print(f"Shape: {X.shape}")

Shape: (12594, 2048)


In [25]:
#stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [26]:
print(f"Training Features (X_train): {X_train.shape}")
print(f"Testing Features (X_test):   {X_test.shape}")
print(f"Training Labels (y_train):   {y_train.shape}")
print(f"Testing Labels (y_test):     {y_test.shape}")

Training Features (X_train): (10075, 2048)
Testing Features (X_test):   (2519, 2048)
Training Labels (y_train):   (10075,)
Testing Labels (y_test):     (2519,)


In [27]:
train_counts = Counter(y_train)
test_counts = Counter(y_test)

#stratification set
print(f"Class 0 in Train: {train_counts[0]}")
print(f"Class 0 in Test:  {test_counts[0]}")
print(f"Ratio: {test_counts[0] / (train_counts[0] + test_counts[0]):.2f}")

Class 0 in Train: 101
Class 0 in Test:  25
Ratio: 0.20


### **1. Logistic Regression**

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

Accuracy: 0.8864628820960698
Precision: 0.8919777288466341
Recall: 0.8864628820960698
F1 Score: 0.8862907920135429


**Conclusion:**
Our simple Logistic Regression baseline achieved **88.6% accuracy and F1 score**. Because Logistic Regression only draws straight mathematical planes, this high score proves that our ResNet50 embeddings successfully separated the 100 insect species into highly distinct clusters.

### **2. Support Vector Machine**

In [32]:
from sklearn.svm import SVC

svm_model = SVC()
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')

print(f"Accuracy: {accuracy_svm}")
print(f"Precision: {precision_svm}")
print(f"Recall: {recall_svm}")
print(f"F1 Score: {f1_svm}")

Accuracy: 0.8265184597062326
Precision: 0.8423138503772177
Recall: 0.8265184597062326
F1 Score: 0.8246173873327365


### **3. Random Forest**

In [30]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf, average='weighted')
recall_rf = recall_score(y_test, y_pred_rf, average='weighted')
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

print(f"Accuracy: {accuracy_rf}")
print(f"Precision: {precision_rf}")
print(f"Recall: {recall_rf}")
print(f"F1 Score: {f1_rf}")

Accuracy: 0.7570464470027789
Precision: 0.7661341832291942
Recall: 0.7570464470027789
F1 Score: 0.7527920438047189


**Model Comparison Conclusion**
We tested a Support Vector Machine (SVM) and a Random Forest Classifier against our Logistic Regression baseline to see how they handle 2,048-dimensional embedding data.

1. Logistic Regression (88.6%):Remained the top performer. Its linear approach is perfectly suited for ResNet50 embeddings, which are inherently linearly separable.
2. SVM (82.6%):Underperformed the baseline. The default RBF kernel attempted to draw non-linear, curved boundaries in a space where simple straight lines are more effective.
3. Random Forest (75.7%): Performed the worst, as expected. Tree-based models evaluate features one axis at a time, which fails to capture the holistic geometric meaning of a dense embedding vector.

### **4. XGBoost**
Let's try this one too

In [33]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb, average='weighted')
recall_xgb = recall_score(y_test, y_pred_xgb, average='weighted')
f1_xgb = f1_score(y_test, y_pred_xgb, average='weighted')

print("\n--- XGBoost Results ---")
print(f"Accuracy: {accuracy_xgb}")
print(f"Precision: {precision_xgb}")
print(f"Recall: {recall_xgb}")
print(f"F1 Score: {f1_xgb}")


--- XGBoost Results ---
Accuracy: 0.7713378324732036
Precision: 0.7764234056708746
Recall: 0.7713378324732036
F1 Score: 0.7700256380557399


**XGBoost Conclusion**
Despite taking 20 minutes to train, XGBoost achieved only 77.1% accuracy, drastically losing to our Logistic Regression baseline. This mathematically proves our theory: complex tree-based models get hopelessly lost in high-dimensional embedding spaces, whereas simple linear models can solve them in seconds.

In [36]:
cm = confusion_matrix(y_test, y_pred)
cm_analysis = cm.copy()

#fill the diagonal (the correct predictions) with zeros
np.fill_diagonal(cm_analysis, 0)
most_confused_count = cm_analysis.max()

# np.unravel_index converts the raw position into the exact Row, Column coordinates
actual_species, predicted_species = np.unravel_index(cm_analysis.argmax(), cm_analysis.shape)

print("CONFUSION MATRIX ANALYSIS (LOGISTIC REGRESSION)")
print(f"The model's biggest repeated mistake happened {most_confused_count} times.")
print(f"It looked at an image of Class {actual_species} but incorrectly guessed Class {predicted_species}.")

CONFUSION MATRIX ANALYSIS (LOGISTIC REGRESSION)
The model's biggest repeated mistake happened 8 times.
It looked at an image of Class 79 but incorrectly guessed Class 36.


**Error Analysis Conclusion**
By analyzing the Confusion Matrix for our Logistic Regression model, we discovered that accuracy alone hides specific blind spots. The model's most frequent error was confusing Class 79 for Class 36 (occurring 8 times). Given the test set size of ~25 images per class, this represents a ~30% failure rate for that specific pair, strongly indicating severe visual similarity or biological mimicry that standard linear boundaries cannot separate.

# **Hyperparameter Tuning**

In [38]:
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.01, 0.1, 1, 10, 100, 1000], 'class_weight': [None, 'balanced']}
GSCV = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=3)
GSCV.fit(X_train, y_train)

print(f"Best Parameters: {GSCV.best_params_}")

Best Parameters: {'C': 10, 'class_weight': None}


In [40]:
best_lr_model = GSCV.best_estimator_
y_pred_tuned = best_lr_model.predict(X_test)

accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned, average='weighted')

print(f"Tuned Accuracy: {accuracy_tuned}")
print(f"Tuned F1 Score: {f1_tuned}")

Tuned Accuracy: 0.8832870186581977
Tuned F1 Score: 0.8829953530651381


In [41]:
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

mistakes_79_as_36 = cm_tuned[79, 36]

#new biggest overall mistake
cm_tuned_analysis = cm_tuned.copy()
np.fill_diagonal(cm_tuned_analysis, 0)
new_max_mistakes = cm_tuned_analysis.max()
new_actual, new_predicted = np.unravel_index(cm_tuned_analysis.argmax(), cm_tuned_analysis.shape)

print("TUNED MODEL DIAGNOSTIC")
print(f"Previous LogReg mistook Class 79 for 36 exactly 8 times.")
print(f"Tuned LogReg mistook Class 79 for 36 exactly {mistakes_79_as_36} times.\n")

print(f"The tuned model's new biggest repeated mistake happened {new_max_mistakes} times.")
print(f"It looked at Class {new_actual} and guessed Class {new_predicted}.")

TUNED MODEL DIAGNOSTIC
Previous LogReg mistook Class 79 for 36 exactly 8 times.
Tuned LogReg mistook Class 79 for 36 exactly 9 times.

The tuned model's new biggest repeated mistake happened 9 times.
It looked at Class 79 and guessed Class 36.


**Hyperparameter Tuning Conclusion**
We performed a Grid Search to tune the `C` (regularization) and `class_weight` parameters of our Logistic Regression model. The tuned model achieved an 88.3% accuracy, a slight decrease from the default baseline's 88.6%. This indicates that Scikit-Learn's default parameters (`C=1.0`) were already optimal for this embedding space, and further tuning caused slight overfitting to the training folds.

## **PCA Experiment**

In [42]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

pca_lr_model = LogisticRegression(max_iter=1000)
pca_lr_model.fit(X_train_pca, y_train)

y_pred_pca = pca_lr_model.predict(X_test_pca)

accuracy_pca = accuracy_score(y_test, y_pred_pca)
f1_pca = f1_score(y_test, y_pred_pca, average='weighted')

print(f"PCA Accuracy: {accuracy_pca}")
print(f"PCA F1 Score: {f1_pca}")

PCA Accuracy: 0.8455736403334657
PCA F1 Score: 0.8450447828235387


**PCA Test Conclusion**
To optimize our model for potential deployment, we used Principal Component Analysis (PCA) to compress the ResNet50 embeddings from 2,048 dimensions down to 100 dimensions (a ~95% reduction in data size).

**Results:**
* Original Logistic Regression (2048D): 88.6% Accuracy
* PCA Logistic Regression (100D): 84.5% Accuracy

**Verdict:** We sacrificed ~4% accuracy to achieve a 20x reduction in feature space. For real-world applications where memory, speed, or compute power are limited (e.g., edge devices or mobile apps), this PCA-compressed model is highly viable.

In [44]:
import joblib

joblib.dump(lr_model, 'best_insect_model_2048D.pkl')
joblib.dump(pca, 'pca_transformer_100D.pkl')
joblib.dump(pca_lr_model, 'best_insect_model_100D.pkl')

print("Models saved")

Models saved
